# 🛰️ NewSpace Analytics: Benchmarking de Mineração de Dados, Tendências de Produção de Informações e Suporte à Decisão

**Domínio de Negócio da Equipe:** *Operações Integradas de Prospecção Mineral de Asteroides Próximos à Terra (NEAs), Gestão de Tráfego em Corredores Orbitais e Subscrição Atuarial Aeroespacial.*

---

### Arquitetura e Governança Científica (Zero Dados Fictícios)
Esta aplicação evolui a modelagem relacional concebida na etapa anterior sobre **100% de bases científicas públicas, oficiais e auditáveis via hash criptográfico SHA-256**, sem qualquer geração sintética ou valores arbitrários:

1. **NASA / Caltech JPL — Small-Body Database (`SBDB`)**: Catálogo completo de **43.072 Objetos Próximos à Terra (NEOs)** com elementos orbitais, incerteza efeméride (`OCC`), arco observacional e taxonomia espectral real (`Bus-DeMeo` / `Tholen`).
2. **NASA / Caltech JPL — `NHATS` (*Near-Earth Object Human/Robotic Accessible Targets*)**: **7.169 asteroides acessíveis** com cálculos balísticos reais de $\Delta v$ mínimo ($\text{km/s}$), duração de missão e janelas viáveis.
3. **USSPACECOM / CelesTrak — Catálogo Oficial `NORAD SATCAT`**: **71.262 objetos espaciais rastreados** desde o Sputnik 1 (`1957–2026`), permitindo classificar fisicamente os corredores orbitais (`LEO`, `MEO`, `GEO`, `HEO/Cislunar`, `Espaço Profundo`) e medir a fração real de satélites ativos vs. detritos espaciais.
4. **Harvard-Smithsonian Center for Astrophysics — `GCAT Launch Log` (J. McDowell)**: Registro histórico completo de **7.418 lançamentos orbitais e interplanetários distintos** distribuídos em **441 modelos de foguetes**, sustentando a modelagem atuarial de confiabilidade (Limite Inferior de Wilson 95% e Bayes Empírico).

**Identidade Visual Oficial do Projeto:** `#160D18` • `#23145B` • `#09456C` • `#026F6E` • `#1CA39E`

In [1]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

ROOT_DIR = Path.cwd().resolve()
if not (ROOT_DIR / "src" / "newspace").exists() and (ROOT_DIR.parent / "src" / "newspace").exists():
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(ROOT_DIR / "src"))

import pandas as pd
import plotly.io as pio
from IPython.display import HTML, Markdown, display

from newspace.config import DEFAULT_CRITERIA, PALETTE
from newspace.ingestion import load_all_real_datasets
from newspace.database import persist_relational_and_parquet, save_actuarial_scenarios
from newspace.benchmarking import run_complete_benchmark_suite
from newspace.selection_trends import (
    analyze_information_production_trends,
    apply_selection_criteria_and_topsis,
)
from newspace.dashboard import (
    create_notebook_interactive_cockpit,
    export_standalone_custom_dashboard,
    fig_actuarial_scenarios_matrix,
    fig_engine_benchmark,
    fig_information_trends,
    fig_ml_benchmark,
    fig_observability_and_corridors,
    fig_topsis_prospecting_map,
)
from newspace.reporting import generate_technical_decision_report

pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

display(HTML(f'''
<div style="display:flex;gap:10px;align-items:center;background:{PALETTE.cosmic_bg};padding:12px 18px;border-radius:8px;border:1px solid {PALETTE.vibrant_cyan};color:#fff;font-family:Segoe UI,sans-serif;">
  <b>Ambiente Inicializado com Sucesso</b> | Raiz: <code>{ROOT_DIR.name}</code> | Paleta Ativa:
  <span style="background:#160D18;padding:3px 10px;border:1px solid #fff;border-radius:4px;">#160D18</span>
  <span style="background:#23145B;padding:3px 10px;border-radius:4px;">#23145B</span>
  <span style="background:#09456C;padding:3px 10px;border-radius:4px;">#09456C</span>
  <span style="background:#026F6E;padding:3px 10px;border-radius:4px;">#026F6E</span>
  <span style="background:#1CA39E;color:#000;font-weight:700;padding:3px 10px;border-radius:4px;">#1CA39E</span>
</div>
'''))

---
## 1. Auditoria de Proveniência (SHA-256), Ingestão Empírica e Persistência Relacional

Nesta etapa validamos criptograficamente os arquivos brutos oficiais em `data/raw/manifest_provenance.json`, realizamos a normalização física das entidades e materializamos a persistência dupla:
- **Banco Relacional Transacional (`SQLite` em modo `WAL` + Índices Compostos Cobertos + `ANALYZE` + Integridade Referencial Estrita)** em `data/processed/newspace_analytics.db`.
- **Camada Analítica Colunar (`Apache Parquet`)** em `data/processed/*.parquet` para avaliação OLAP.

In [2]:
datasets = load_all_real_datasets()
db_counts = persist_relational_and_parquet(datasets)

prov_df = pd.DataFrame(datasets.provenance["datasets"])[
    ["id", "description", "size_mb", "sha256", "verified_at_utc"]
].copy()
prov_df["sha256"] = prov_df["sha256"].str.slice(0, 16) + "..."

display(Markdown("### 1.1 Manifesto de Proveniência Científica Auditada"))
display(prov_df)

summary_entities = pd.DataFrame(
    [{"Tabela Relacional (SQLite & Parquet)": k, "Registros Empíricos Reais": f"{v:,}"} for k, v in db_counts.items()]
)
display(Markdown("### 1.2 Volumetria Persistida com Integridade Referencial Verificada (`PRAGMA foreign_key_check`)"))
display(summary_entities)

### 1.1 Manifesto de Proveniência Científica Auditada

,id,description,size_mb,sha256,verified_at_utc
0,nasa_jpl_nhats,NASA/Caltech JPL NHATS Accessible Near-Earth A...,2.447,c3850af9adb913c3...,2026-11-04T20:57:54.614156+00:00
1,nasa_jpl_sbdb_neo,NASA/Caltech JPL Small-Body Database (All Near...,6.990,de1c64180c723811...,2026-11-04T20:57:59.904676+00:00
2,celestrak_norad_satcat,CelesTrak / USSPACECOM Official NORAD Satellit...,6.478,5361f54feabc637a...,2026-11-04T20:58:08.739974+00:00
3,gcat_orbital_launchlog,Harvard-Smithsonian CfA GCAT Historical Orbita...,5.037,14dba638398fad33...,2026-11-04T20:58:10.281331+00:00


### 1.2 Volumetria Persistida com Integridade Referencial Verificada (`PRAGMA foreign_key_check`)

,Tabela Relacional (SQLite & Parquet),Registros Empíricos Reais
0,VEICULO_LANCADOR,441
1,HISTORICO_LANCAMENTO,"7,418"
2,CORREDOR_ORBITAL,7
3,OBJETO_TRAFEGO_ORBITAL,"71,262"
4,ASTEROIDE_PROSPECCAO,"43,072"


---
## 2. Benchmarking Multidimensional da Mineração e Persistência de Dados

Executamos uma bateria quantitativa rigorosa em **três dimensões complementares**:
1. **Motores de Persistência e Consulta (`OLTP` vs. `OLAP`)**: Comparação de latência mediana, P95, pico de memória (`tracemalloc`), *throughput* e taxa de compressão entre `SQLite (Sem Índices)`, `SQLite (Indexado + WAL)`, `Pandas (In-Memory)`, `Polars (Lazy Parquet)` e `DuckDB (OLAP Parquet)` sob três cargas reais (`W1`: Join de Tráfego Orbital 71k linhas; `W2`: Filtragem Multicritério NASA NHATS 43k linhas; `W3`: Série Histórica Atuarial 7,4k lançamentos).
2. **Mineração Não-Supervisionada (Segmentação Astrodinâmica)**: Avaliação de 5 algoritmos de clusterização sobre os **7.148 asteroides acessíveis NHATS** com métricas `Silhouette`, `Calinski-Harabasz`, `Davies-Bouldin` e custo computacional.
3. **Mineração Supervisionada Preditiva (Risco Atuarial de Lançamento)**: Avaliação em validação cruzada estratificada (`5-Fold CV`) de 5 classificadores sobre **7.418 lançamentos orbitais**, utilizando engenharia de atributos **estritamente cronológica/causal (zero vazamento de futuro)** e priorizando `ROC-AUC` e `Brier Score` (calibração probabilística de sinistro).

In [3]:
bench = run_complete_benchmark_suite(datasets)

display(Markdown("### 2.1 Síntese Global dos Motores de Persistência e Consulta Analítica"))
display(bench.engine_summary)

display(Markdown("### 2.2 Detalhamento por Carga de Trabalho (Workloads W1, W2 e W3 — Latência Mediana ms)"))
display(
    bench.engine_workloads.pivot(
        index="engine", columns="workload", values="median_ms"
    ).round(3)
)

display(Markdown("### 2.3 Eficiência de Armazenamento e Compressão em Disco"))
display(bench.storage_footprint)

fig_engine_benchmark(bench).show()

### 2.1 Síntese Global dos Motores de Persistência e Consulta Analítica

,engine,latencia_total_mediana_ms,latencia_media_ms,p95_medio_ms,pico_memoria_max_kb,throughput_medio_krows_s,speedup_vs_sqlite_sem_indice
0,Polars (Lazy Parquet),10.922,3.600,3.779,5.0,11008.5,3.88
1,SQLite (Indexado + WAL),17.371,5.789,5.829,20.7,27270.0,2.44
2,DuckDB (OLAP Parquet),18.344,6.140,6.442,218.8,6239.9,2.31
3,Pandas (In-Memory),34.314,11.430,11.584,4260.8,3605.2,1.23
4,SQLite (Sem Índices),42.371,14.160,14.346,20.7,2791.3,1.00


### 2.2 Detalhamento por Carga de Trabalho (Workloads W1, W2 e W3 — Latência Mediana ms)

workload,W1_Join_Trafego_Orbital,W2_Prospeccao_Asteroides_NHATS,W3_Serie_Historica_Atuarial
engine,,,
DuckDB (OLAP Parquet),8.626,4.815,4.903
Pandas (In-Memory),16.618,7.419,10.277
Polars (Lazy Parquet),4.388,2.924,3.610
SQLite (Indexado + WAL),14.944,0.590,1.837
SQLite (Sem Índices),26.113,13.116,3.142


### 2.3 Eficiência de Armazenamento e Compressão em Disco

,formato_persistencia,tamanho_mb,registros_totais,razao_compressao_vs_bruto
0,Arquivos Brutos Originais (JSON + CSV + TSV),20.953,122200,1.00
1,SQLite Relacional Normalizado (Sem Índices),18.086,122200,1.16
2,SQLite Relacional Normalizado + Índices B-Tree,23.352,122200,0.90
3,Apache Parquet Colunar (Compressão ZSTD/Snappy),4.301,122200,4.87


In [4]:
display(Markdown("### 2.4 Benchmarking de Clusterização Não-Supervisionada (Asteroides NASA NHATS)"))
display(bench.clustering_benchmark)

cluster_profiles = (
    bench.clustered_nhats.groupby("arquetipo_astrodinamico", as_index=False)
    .agg(
        qtd_asteroides=("spkid", "count"),
        delta_v_mediano_kms=("min_delta_v_kms", "median"),
        duracao_mediana_dias=("duracao_min_dv_dias", "median"),
        excentricidade_mediana=("excentricidade_e", "median"),
        observacoes_medianas=("num_observacoes", "median"),
    )
    .round(3)
)
display(Markdown("#### Perfil Físico dos Arquétipos Minerados Automaticamente"))
display(cluster_profiles)

display(Markdown("### 2.5 Benchmarking de Classificadores Supervisionados Atuariais (5-Fold CV)"))
display(bench.classification_benchmark)

fig_ml_benchmark(bench).show()

### 2.4 Benchmarking de Clusterização Não-Supervisionada (Asteroides NASA NHATS)

,algoritmo,amostras_nhats,silhouette_score,calinski_harabasz,davies_bouldin,tempo_execucao_ms,pico_memoria_kb
0,Bisecting K-Means (k=4),7148,0.2326,2046.7,1.3394,27.91,1356.3
1,"K-Means (Elkan, k=4)",7148,0.2322,2141.6,1.3189,189.39,1330.4
2,Agglomerative Ward (k=4),7148,0.1929,1630.4,1.2465,1366.53,225157.7
3,MiniBatch K-Means (k=4),7148,0.1851,2005.0,1.5065,42.86,942.1
4,"Gaussian Mixture (GMM Full, k=4)",7148,0.0923,1372.7,2.0051,239.76,1889.3


#### Perfil Físico dos Arquétipos Minerados Automaticamente

,arquetipo_astrodinamico,qtd_asteroides,delta_v_mediano_kms,duracao_mediana_dias,excentricidade_mediana,observacoes_medianas
0,Cluster A: Órbita Quase-Terrestre (Baixo Δv / ...,4369,8.989,362.0,0.244,33.0
1,Cluster B: NEAs Moderados Bem Caracterizados (...,940,9.359,426.0,0.258,43.0
2,Cluster C: NEAs Energéticos / Excentricidade M...,271,9.486,194.0,0.255,44.0
3,Cluster D: Alvos de Alta Energia / Excentricid...,1568,10.329,370.0,0.189,45.0


### 2.5 Benchmarking de Classificadores Supervisionados Atuariais (5-Fold CV)

,modelo_atuarial,lancamentos_avaliados,taxa_base_sinistro_pct,roc_auc,pr_auc,brier_score_calibracao,log_loss,f1_weighted,tempo_cv_5fold_ms
0,Regressão Logística L2 (Baseline Atuarial),7418,7.28,0.7846,0.3093,0.05851,0.2175,0.9032,54.4
1,Extra Trees (200 Árvores),7418,7.28,0.7762,0.3354,0.05750,0.2160,0.9036,1470.7
2,Rede Neural MLP (64x32 ReLU),7418,7.28,0.7707,0.2853,0.05973,0.2231,0.9036,2234.9
3,Random Forest (200 Árvores),7418,7.28,0.7678,0.3184,0.05809,0.2182,0.9102,2180.4
4,HistGradientBoosting (GBDT),7418,7.28,0.7576,0.3113,0.05876,0.2263,0.9121,1133.0


---
## 3. Tendência de Produção de Informações Espaciais (1957–2030) e Gargalos de Negócio

A análise temporal conjunta das três fontes empíricas demonstra:
1. **Mudança Estrutural Exponencial pós-2015 (*Era NewSpace*)**: Crescimento abrupto na produção anual de telemetria orbital (`PAY`), capacidade de lançamento compartilhado (*rideshare* / cargas por foguete) e descoberta astrométrica de NEOs.
2. **Lei Empírica Informação $\times$ Risco Navegacional**: Correlação estatística de Spearman comprovando que o acúmulo de observações astrométricas colapsa a incerteza orbital (`OCC 9 -> OCC 0`), transformando corpos recém-descobertos em alvos seguráveis.
3. **Bifurcação de Detritos nos Corredores Orbitais & Gargalo Espectroscópico**: Identificação da concentração crítica de objetos inertes (`DEB + R/B`) nos corredores `LEO_SSO_ALTA` e `HEO_CISLUNAR`, bem como da escassez de espectroscopia mineralógica nos NEOs catalogados.

In [5]:
trends = analyze_information_production_trends(datasets)

display(Markdown("### 3.1 Comparativo Estrutural entre Eras Espaciais (Anos Completos 1957–2025)"))
display(trends.era_comparison)

display(Markdown("### 3.2 Projeção Log-Linear da Produção de Informações (2026–2030)"))
display(trends.forecast_2030)

rho_occ, pval_occ = trends.spearman_obs_vs_occ
gap = trends.characterization_gap
display(Markdown(
    f"**Correlação Empírica Observações vs. Incerteza Orbital (OCC):** Spearman $\\rho = {rho_occ:.4f}$ ($p = {pval_occ:.2e}$)  \n"
    f"**Diagnóstico de Caracterização Espectral NASA JPL:** Apenas **{gap['pct_neos_com_espectro']:.2f}%** "
    f"({gap['neos_com_espectro_real']:,} de {gap['total_neos_catalogados']:,} NEOs) possuem classe espectral medida, "
    f"evidenciando a demanda imediata por missões precursoras de sensoriamento mineral."
))

fig_information_trends(trends).show()

### 3.1 Comparativo Estrutural entre Eras Espaciais (Anos Completos 1957–2025)

,era_espacial,anos_completos,media_anual_obj_orbitais,media_anual_cargas_pay,media_anual_neos_descobertos,media_cargas_por_foguete,taxa_sucesso_media_pct,indice_info_medio
0,1. Era Estatal Clássica (1957–1990),34,783.18,121.91,12.09,1.71,86.09,49.29
1,2. Transição Comercial (1991–2014),24,804.42,136.46,544.92,1.98,93.80,83.45
2,3. Era NewSpace Exponencial (2015–2026),11,1949.73,1563.36,2475.73,8.87,94.81,340.82


### 3.2 Projeção Log-Linear da Produção de Informações (2026–2030)

,ano,indice_info_projetado,ic95_inferior,ic95_superior,objetos_orbitais_ano_proj,neos_descobertos_ano_proj,r2_ajuste_loglinear,cagr_newspace_2015_2025_pct
0,2026,800.8,654.3,980.1,7464,3574,0.9691,16.95
1,2027,944.5,771.7,1156.0,9931,3814,0.9691,16.95
2,2028,1114.0,910.2,1363.5,13213,4071,0.9691,16.95
3,2029,1314.0,1073.6,1608.2,17580,4346,0.9691,16.95
4,2030,1549.8,1266.3,1896.9,23390,4638,0.9691,16.95


**Correlação Empírica Observações vs. Incerteza Orbital (OCC):** Spearman $\rho = -0.7274$ ($p = 0.00e+00$)  
**Diagnóstico de Caracterização Espectral NASA JPL:** Apenas **0.76%** (327 de 43,072 NEOs) possuem classe espectral medida, evidenciando a demanda imediata por missões precursoras de sensoriamento mineral.

In [6]:
display(Markdown("### 3.3 Diagnóstico Físico dos Corredores Orbitais (NORAD SATCAT Atual)"))
display(
    datasets.corridors[
        [
            "id_corredor",
            "regime_orbital",
            "objetos_em_orbita",
            "satelites_ativos",
            "detritos_e_estagios_orbita",
            "razao_detritos_pct",
            "expansao_pos_2021_pct",
            "indice_congestao_orbital",
        ]
    ]
)

fig_observability_and_corridors(datasets, trends).show()

### 3.3 Diagnóstico Físico dos Corredores Orbitais (NORAD SATCAT Atual)

,id_corredor,regime_orbital,objetos_em_orbita,satelites_ativos,detritos_e_estagios_orbita,razao_detritos_pct,expansao_pos_2021_pct,indice_congestao_orbital
0,LEO_VLEO,LEO,2316,1916,309,13.34,15.87,15.62
1,LEO_CONSTELACAO,LEO,13483,12599,697,5.17,76.49,61.94
2,LEO_SSO_ALTA,LEO,13511,1991,10072,74.55,17.93,89.82
3,MEO_NAVEGACAO,MEO,2625,193,2111,80.42,4.64,43.83
4,GEO_CINTURAO,GEO,1793,633,412,22.98,11.92,17.15
5,HEO_CISLUNAR,CISLUNAR,1503,73,1262,83.97,5.75,40.26
6,ESPACO_PROFUNDO,HELIOCENTRICO,258,62,116,44.96,16.94,19.13


---
## 4. Critérios de Seleção de Dados do Negócio (MCDA-TOPSIS) e Matriz Atuarial

Aplicamos os **Critérios de Seleção de Dados** alinhados ao modelo de negócio integrado da equipe:
- **Acessibilidade Energética e Operacional:** Solução balística viável no catálogo `NASA NHATS`, $\Delta v \le 6,5\text{ km/s}$ e duração $\le 450\text{ dias}$.
- **Qualidade e Maturidade da Informação Orbital:** Código de Incerteza `OCC <= 3` e mínimo de `50` observações astrométricas confirmadas.
- **Porte Físico Mínimo:** Diâmetro físico de referência $\ge 25\text{ m}$.
- **Subscrição Atuarial de Lançadores:** Mínimo de `10` voos orbitais, operação ativa na era NewSpace e Limite Inferior de Confiabilidade de Wilson (95%) $\ge 85\%$.
- **Ordenação Multicritério (`TOPSIS`)**: Combina eficiência energética ($\Delta v$ e duração), índice composto de maturidade informacional, escala física e redundância de trajetórias viáveis.

In [7]:
selection = apply_selection_criteria_and_topsis(datasets)
n_saved_scen = save_actuarial_scenarios(selection.actuarial_scenarios)

display(Markdown("### 4.1 Funil de Seletividade dos Critérios de Negócio"))
display(selection.funnel_stages)

display(Markdown("### 4.2 Top 15 Asteroides Próximos à Terra Recomendados (Ranking MCDA-TOPSIS)"))
display(
    selection.ranked_asteroids[
        [
            "ranking_topsis",
            "spkid",
            "nome_completo",
            "classe_espectral",
            "grupo_mineralogico",
            "min_delta_v_kms",
            "duracao_min_dv_dias",
            "diametro_fisico_ref_m",
            "num_observacoes",
            "codigo_incerteza_occ",
            "indice_maturidade_info",
            "score_topsis",
        ]
    ].head(15)
)

fig_topsis_prospecting_map(selection).show()

### 4.1 Funil de Seletividade dos Critérios de Negócio

,etapa_funil,candidatos,retencao_global_pct
0,1. Catálogo Global NASA SBDB (NEOs),43072,100.000
1,2. Solução Balística Viável (NASA NHATS),7169,16.644
2,3. Janela Energética (Δv ≤ 6.5 km/s & Duração ...,1042,2.419
3,4. Qualidade Informacional (OCC ≤ 3 & Obs ≥ 50),174,0.404
4,5. Porte Físico Operacional (Diâmetro Ref ≥ 25 m),92,0.214
5,6. Portfólio Qualificado Final (Pós-Filtro Esp...,92,0.214


### 4.2 Top 15 Asteroides Próximos à Terra Recomendados (Ranking MCDA-TOPSIS)

,ranking_topsis,spkid,nome_completo,classe_espectral,grupo_mineralogico,min_delta_v_kms,duracao_min_dv_dias,diametro_fisico_ref_m,num_observacoes,codigo_incerteza_occ,indice_maturidade_info,score_topsis
0,1,20099942,99942 Apophis (2004 MN4),Sq,Complexo-S (Silicatado / Metais & Rochoso),6.049,354.0,340.000000,7370,0.0,98.89,60.62
1,2,50457409,(2009 HC),NAO_ESPECTROGRAFADO,Não Caracterizado (Alvo de Sonda Precursora),4.440,362.0,42.661458,209,0.0,69.67,57.25
2,3,50674005,(2014 MF18),NAO_ESPECTROGRAFADO,Não Caracterizado (Alvo de Sonda Precursora),6.269,202.0,69.000000,78,0.0,67.59,54.10
3,4,20341843,341843 (2008 EV5),NAO_ESPECTROGRAFADO,Não Caracterizado (Alvo de Sonda Precursora),6.291,362.0,400.000000,1486,0.0,82.61,51.52
4,5,50315027,(2006 BZ147),NAO_ESPECTROGRAFADO,Não Caracterizado (Alvo de Sonda Precursora),4.091,362.0,30.166206,64,3.0,50.40,50.56
5,6,54421069,(2024 BF),NAO_ESPECTROGRAFADO,Não Caracterizado (Alvo de Sonda Precursora),4.546,362.0,46.195238,82,2.0,53.53,46.42
6,7,20478784,478784 (2012 UV136),NAO_ESPECTROGRAFADO,Não Caracterizado (Alvo de Sonda Precursora),4.670,394.0,27.694765,148,0.0,66.55,46.39
7,8,54017581,(2020 MU1),NAO_ESPECTROGRAFADO,Não Caracterizado (Alvo de Sonda Precursora),4.403,370.0,40.187063,54,2.0,49.95,45.58
8,9,54360089,(2023 KX3),NAO_ESPECTROGRAFADO,Não Caracterizado (Alvo de Sonda Precursora),4.170,370.0,25.690465,80,3.0,42.33,45.25
9,10,20613862,613862 (2007 UY1),NAO_ESPECTROGRAFADO,Não Caracterizado (Alvo de Sonda Precursora),5.579,362.0,88.622796,312,0.0,70.84,44.37


In [8]:
display(Markdown("### 4.3 Veículos Lançadores Qualificados pelo Critério Atuarial (Wilson 95%)"))
display(
    selection.qualified_launchers[
        [
            "id_veiculo",
            "modelo_veiculo",
            "agencia_principal",
            "pais_estado",
            "total_lancamentos",
            "taxa_sucesso_pct",
            "wilson_lower_95_pct",
            "confiabilidade_bayes_pct",
            "ultimo_voo_ano",
        ]
    ].head(10)
)

display(Markdown(f"### 4.4 Cenários Integrados de Subscrição Atuarial ({n_saved_scen} cenários persistidos no SQLite)"))
display(
    selection.actuarial_scenarios[
        [
            "id_cenario",
            "nome_asteroide",
            "modelo_veiculo",
            "id_corredor_transito",
            "min_delta_v_kms",
            "indice_maturidade_info",
            "confiabilidade_wilson_pct",
            "score_topsis",
            "score_risco_combinado",
            "rating_atuarial",
        ]
    ].head(12)
)

fig_actuarial_scenarios_matrix(selection).show()

### 4.3 Veículos Lançadores Qualificados pelo Critério Atuarial (Wilson 95%)

,id_veiculo,modelo_veiculo,agencia_principal,pais_estado,total_lancamentos,taxa_sucesso_pct,wilson_lower_95_pct,confiabilidade_bayes_pct,ultimo_voo_ano
0,LV_0001,Falcon 9,SPX,US,699,99.57,98.75,99.52,2026
1,LV_0012,Chang Zheng 2D,SAST,CN,102,99.02,94.65,98.73,2026
2,LV_0019,Chang Zheng 2C,CALT,CN,74,98.65,92.73,98.27,2026
3,LV_0022,Soyuz-FG,RAKA,RU,70,98.57,92.34,98.18,2019
4,LV_0025,Chang Zheng 4B,SAST,CN,59,98.31,91.00,97.87,2026
5,LV_0021,Ariane 5ECA,AE,F,72,97.22,90.43,96.93,2019
6,LV_0017,Soyuz-2-1B,STSM,F,87,96.55,90.35,96.34,2026
7,LV_0038,H-IIA 202,NASDA,J,35,100.00,90.11,99.09,2025
8,LV_0018,Soyuz-2-1A,STSM,F,82,96.34,89.79,96.13,2026
9,LV_0013,Chang Zheng 3B,CALT,CN,102,95.10,89.03,94.99,2026


### 4.4 Cenários Integrados de Subscrição Atuarial (75 cenários persistidos no SQLite)

,id_cenario,nome_asteroide,modelo_veiculo,id_corredor_transito,min_delta_v_kms,indice_maturidade_info,confiabilidade_wilson_pct,score_topsis,score_risco_combinado,rating_atuarial
0,NS_SCEN_001,99942 Apophis (2004 MN4),Falcon 9,HEO_CISLUNAR,6.049,98.89,98.75,60.62,24.07,A
1,NS_SCEN_002,99942 Apophis (2004 MN4),Chang Zheng 2D,HEO_CISLUNAR,6.049,98.89,94.65,60.62,31.24,BB
2,NS_SCEN_003,99942 Apophis (2004 MN4),Chang Zheng 2C,HEO_CISLUNAR,6.049,98.89,92.73,60.62,34.60,BB
3,NS_SCEN_004,(2009 HC),Falcon 9,HEO_CISLUNAR,4.440,69.67,98.75,57.25,29.59,A
4,NS_SCEN_005,(2009 HC),Chang Zheng 2D,HEO_CISLUNAR,4.440,69.67,94.65,57.25,36.76,BB
5,NS_SCEN_006,(2009 HC),Chang Zheng 2C,HEO_CISLUNAR,4.440,69.67,92.73,57.25,40.12,B
6,NS_SCEN_007,(2014 MF18),Falcon 9,HEO_CISLUNAR,6.269,67.59,98.75,54.10,29.46,A
7,NS_SCEN_008,(2014 MF18),Chang Zheng 2D,HEO_CISLUNAR,6.269,67.59,94.65,54.10,36.63,BB
8,NS_SCEN_009,(2014 MF18),Chang Zheng 2C,HEO_CISLUNAR,6.269,67.59,92.73,54.10,39.99,BB
9,NS_SCEN_010,341843 (2008 EV5),Falcon 9,HEO_CISLUNAR,6.291,82.61,98.75,51.52,28.57,A


---
## 5. Dashboard Customizável (Simulador Interativo Jupyter + Cockpit Web HTML5 Standalone)

Abaixo você dispõe de **duas interfaces interativas sincronizadas**:
1. **Cockpit Interativo `ipywidgets` no próprio Notebook**: Altere dinamicamente os limites de $\Delta v$, duração, incerteza `OCC`, observações mínimas, porte físico e os **4 pesos decisórios do TOPSIS** para recalcular em tempo real o portfólio qualificado.
2. **Aplicação Web Standalone Customizável (`reports/newspace_dashboard.html`)**: Exportada automaticamente com motor JavaScript TOPSIS no cliente sobre todos os 7.169 alvos NHATS — pode ser aberta diretamente em qualquer navegador sem servidor ativo e permite exportar CSVs customizados.

In [9]:
dashboard_html_path = export_standalone_custom_dashboard(datasets, bench, trends, selection)
display(Markdown(f"✅ **Cockpit Web Customizável exportado com sucesso para:** `{dashboard_html_path.relative_to(ROOT_DIR)}`"))

cockpit_widget = create_notebook_interactive_cockpit(datasets)
display(cockpit_widget)

,ranking_topsis,pdes,nome_completo,classe_espectral,min_delta_v_kms,duracao_min_dv_dias,diametro_fisico_ref_m,num_observacoes,codigo_incerteza_occ,score_topsis
0,1,99942,99942 Apophis (2004 MN4),Sq,6.049,354.0,340.000000,7370,0.0,60.62
1,2,2009 HC,(2009 HC),NAO_ESPECTROGRAFADO,4.440,362.0,42.661458,209,0.0,57.25
2,3,2014 MF18,(2014 MF18),NAO_ESPECTROGRAFADO,6.269,202.0,69.000000,78,0.0,54.10
3,4,341843,341843 (2008 EV5),NAO_ESPECTROGRAFADO,6.291,362.0,400.000000,1486,0.0,51.52
4,5,2006 BZ147,(2006 BZ147),NAO_ESPECTROGRAFADO,4.091,362.0,30.166206,64,3.0,50.56
5,6,2024 BF,(2024 BF),NAO_ESPECTROGRAFADO,4.546,362.0,46.195238,82,2.0,46.42
6,7,478784,478784 (2012 UV136),NAO_ESPECTROGRAFADO,4.670,394.0,27.694765,148,0.0,46.39
7,8,2020 MU1,(2020 MU1),NAO_ESPECTROGRAFADO,4.403,370.0,40.187063,54,2.0,45.58
8,9,2023 KX3,(2023 KX3),NAO_ESPECTROGRAFADO,4.170,370.0,25.690465,80,3.0,45.25
9,10,613862,613862 (2007 UY1),NAO_ESPECTROGRAFADO,5.579,362.0,88.622796,312,0.0,44.37


---
## 6. Geração Automatizada do Relatório Técnico Executivo para Tomada de Decisão

Consolidamos automaticamente todas as métricas empíricas obtidas na execução em um **Relatório Técnico Executivo** parametrizado (salvo em `reports/relatorio_tecnico_decisao.md` e em versão formatada para apresentação/impressão PDF em `reports/relatorio_tecnico_decisao.html`), exibido integralmente abaixo:

In [10]:
md_report_path, html_report_path = generate_technical_decision_report(
    datasets, bench, trends, selection
)
display(Markdown(
    f"📄 **Relatórios gerados:** `{md_report_path.relative_to(ROOT_DIR)}` | `{html_report_path.relative_to(ROOT_DIR)}`"
))
display(Markdown("---"))
display(Markdown(md_report_path.read_text(encoding="utf-8")))

📄 **Relatórios gerados:** `reports\relatorio_tecnico_decisao.md` | `reports\relatorio_tecnico_decisao.html`

---

# Relatório Técnico Executivo para Tomada de Decisão — Operações NewSpace

**Data de Geração:** 04/11/2026 18:23  
**Escopo do Negócio:** Prospecção Mineral de Asteroides Próximos à Terra (NEAs), Gestão de Risco em Corredores Orbitais e Subscrição Atuarial Aeroespacial  
**Política de Integridade Científica:** 100% Dados Públicos Reais Auditados via SHA-256 (122,200 registros empíricos processados — zero registros sintéticos ou fictícios)

---

## 1. Sumário Executivo e Proveniência Empírica

Este relatório consolida os resultados quantitativos do **Benchmarking de Mineração e Persistência de Dados**, da **Modelagem de Tendências de Produção de Informações (1957–2030)** e da **Seleção Multicritério (MCDA-TOPSIS) integrada ao Risco Atuarial** para orientar decisões estratégicas de investimento, engenharia de missão e subscrição de apólices no ecossistema NewSpace.

| Base Oficial Auditada | Instituição Mantenedora | Volume Real Processado | Escopo Técnico |
| :--- | :--- | ---: | :--- |
| **NORAD SATCAT** | USSPACECOM / CelesTrak | 71,262 objetos | Rastreamento orbital completo (1957–2026), status operacional, apogeu/perigeu e detritos |
| **Small-Body DB (SBDB)** | NASA / Caltech JPL | 43,072 asteroides | Elementos orbitais, incerteza efeméride (OCC), arco observacional e taxonomia espectral |
| **NHATS Trajectory DB** | NASA / Caltech JPL | 7,169 alvos viáveis | Soluções balísticas de ida e volta, $\Delta v$ mínimo (km/s) e janelas de lançamento |
| **GCAT Launch Log** | Harvard-Smithsonian CfA | 7,418 lançamentos (441 modelos) | Histórico completo de lançamentos orbitais e de espaço profundo desde o Sputnik 1 |

---

## 2. Diagnóstico do Benchmarking da Mineração e Persistência de Dados

### 2.1 Arquitetura de Persistência e Motores de Consulta Analítica (OLTP vs OLAP)

A evolução do modelo relacional da Fase 1 foi avaliada frente a motores analíticos vetorizados modernos em três cargas representativas de negócio (joins de tráfego orbital, filtragem astrodinâmica NHATS e agregação histórica atuarial):

| engine                  |   latencia_total_mediana_ms |   latencia_media_ms |   p95_medio_ms |   pico_memoria_max_kb |   throughput_medio_krows_s |   speedup_vs_sqlite_sem_indice |
|:------------------------|----------------------------:|--------------------:|---------------:|----------------------:|---------------------------:|-------------------------------:|
| Polars (Lazy Parquet)   |                      10.922 |               3.6   |          3.779 |                   5   |                    11008.5 |                           3.88 |
| SQLite (Indexado + WAL) |                      17.371 |               5.789 |          5.829 |                  20.7 |                    27270   |                           2.44 |
| DuckDB (OLAP Parquet)   |                      18.344 |               6.14  |          6.442 |                 218.8 |                     6239.9 |                           2.31 |
| Pandas (In-Memory)      |                      34.314 |              11.43  |         11.584 |                4260.8 |                     3605.2 |                           1.23 |
| SQLite (Sem Índices)    |                      42.371 |              14.16  |         14.346 |                  20.7 |                     2791.3 |                           1    |

**Conclusões de Engenharia de Dados:**
- **Motor Analítico Vencedor:** **Polars (Lazy Parquet)** atingiu latência total mediana de **10.92 ms** (**3.88x mais rápido** que o SQLite relacional sem índices).
- **Validação da Indexação Relacional:** A adição de índices compostos B-Tree e modo `WAL` no SQLite acelerou as consultas relacionais em **2.44x** (42.37 ms $\rightarrow$ 17.37 ms), validando o SQLite indexado para persistência transacional (ACID + chaves estrangeiras) em conjunto com **Apache Parquet (4.30 MB vs 20.95 MB brutos — compressão de 4.87x)** para analytics em tempo real.

### 2.2 Benchmarking de Algoritmos de Mineração de Dados (Clusterização e Risco Atuarial)

#### A. Segmentação Não-Supervisionada de Asteroides Acessíveis (NASA NHATS)
| algoritmo                        |   amostras_nhats |   silhouette_score |   calinski_harabasz |   davies_bouldin |   tempo_execucao_ms |   pico_memoria_kb |
|:---------------------------------|-----------------:|-------------------:|--------------------:|-----------------:|--------------------:|------------------:|
| Bisecting K-Means (k=4)          |             7148 |             0.2326 |              2046.7 |           1.3394 |               27.91 |            1356.3 |
| K-Means (Elkan, k=4)             |             7148 |             0.2322 |              2141.6 |           1.3189 |              189.39 |            1330.4 |
| Agglomerative Ward (k=4)         |             7148 |             0.1929 |              1630.4 |           1.2465 |             1366.53 |          225158   |
| MiniBatch K-Means (k=4)          |             7148 |             0.1851 |              2005   |           1.5065 |               42.86 |             942.1 |
| Gaussian Mixture (GMM Full, k=4) |             7148 |             0.0923 |              1372.7 |           2.0051 |              239.76 |            1889.3 |

- O algoritmo **Bisecting K-Means (k=4)** obteve a maior coesão estrutural (**Silhouette = 0.2326**, **Davies-Bouldin = 1.3394** em **27.9 ms**), isolando o agrupamento de alvos de órbita quase-terrestre de baixo $\Delta v$ dos objetos de alta excentricidade.

#### B. Modelagem Preditiva Supervisionada de Sinistro de Lançamento (Validação Cruzada Estratificada 5-Fold, Sem Vazamento Temporal)
| modelo_atuarial                            |   lancamentos_avaliados |   taxa_base_sinistro_pct |   roc_auc |   pr_auc |   brier_score_calibracao |   log_loss |   f1_weighted |   tempo_cv_5fold_ms |
|:-------------------------------------------|------------------------:|-------------------------:|----------:|---------:|-------------------------:|-----------:|--------------:|--------------------:|
| Regressão Logística L2 (Baseline Atuarial) |                    7418 |                     7.28 |    0.7846 |   0.3093 |                  0.05851 |     0.2175 |        0.9032 |                54.4 |
| Extra Trees (200 Árvores)                  |                    7418 |                     7.28 |    0.7762 |   0.3354 |                  0.0575  |     0.216  |        0.9036 |              1470.7 |
| Rede Neural MLP (64x32 ReLU)               |                    7418 |                     7.28 |    0.7707 |   0.2853 |                  0.05973 |     0.2231 |        0.9036 |              2234.9 |
| Random Forest (200 Árvores)                |                    7418 |                     7.28 |    0.7678 |   0.3184 |                  0.05809 |     0.2182 |        0.9102 |              2180.4 |
| HistGradientBoosting (GBDT)                |                    7418 |                     7.28 |    0.7576 |   0.3113 |                  0.05876 |     0.2263 |        0.9121 |              1133   |

- **Modelo Atuarial Recomendado:** **Regressão Logística L2 (Baseline Atuarial)** entregou **ROC-AUC = 0.7846** e **Brier Score = 0.05851** (calibração probabilística essencial para precificação de prêmios securitários).
- **Principal Fator Empírico de Risco:** *Taxa Histórica Prévia do Foguete (Bayes)* responde por **18.74%** do poder explicativo do modelo, demonstrando que herança real acumulada e histórico prévio superam variáveis isoladas de porte de carga.

---

## 3. Tendência de Produção de Informações e Gargalos Estratégicos (1957–2030)

A comparação estrutural entre as três eras espaciais comprova uma mudança de regime exponencial na geração de telemetria orbital e astrometria de espaço profundo:

| era_espacial                            |   anos_completos |   media_anual_obj_orbitais |   media_anual_cargas_pay |   media_anual_neos_descobertos |   media_cargas_por_foguete |   taxa_sucesso_media_pct |   indice_info_medio |
|:----------------------------------------|-----------------:|---------------------------:|-------------------------:|-------------------------------:|---------------------------:|-------------------------:|--------------------:|
| 1. Era Estatal Clássica (1957–1990)     |               34 |                     783.18 |                   121.91 |                          12.09 |                       1.71 |                    86.09 |               49.29 |
| 2. Transição Comercial (1991–2014)      |               24 |                     804.42 |                   136.46 |                         544.92 |                       1.98 |                    93.8  |               83.45 |
| 3. Era NewSpace Exponencial (2015–2026) |               11 |                    1949.73 |                  1563.36 |                        2475.73 |                       8.87 |                    94.81 |              340.82 |

### Principais Evidências Quantitativas para Tomada de Decisão:
1. **Explosão de Telemetria na Era NewSpace (2015–2026):** A média anual de objetos orbitais catalogados saltou de **783/ano** na Era Clássica para **1950/ano** na Era NewSpace, enquanto a descoberta anual de asteroides próximos à Terra cresceu de **12.1/ano** para **2476/ano**.
2. **Projeção 2030 (Ajuste Log-Linear, $R^2 = 0.9691$, CAGR 2015–2025 = 16.95% a.a.):** O Índice Composto de Produção de Informações (Base 2010 = 100) projeta atingir **1549.8** até 2030 (IC 95%: [1266.3, 1896.9]), exigindo arquiteturas OLAP colunares (Polars/DuckDB) para ingestão contínua de efemérides e telemetria SSA (*Space Situational Awareness*).
3. **Lei Empírica Informação $\times$ Risco Orbital:** Verificou-se correlação monotônica altamente significativa (**Spearman $\rho = -0.7274$**) entre o volume de observações astrométricas e a redução do código de incerteza orbital da NASA (`OCC`). Mais observações convertem diretamente risco epistêmico em precisão navegacional segurável.
4. **O Grande Gargalo Espectroscópico (Oportunidade de Negócio Imediata):** Embora existam **43,072 NEOs** com órbitas rastreadas (7,169 acessíveis via NHATS), **apenas 0.76% (327 objetos)** possuem espectroscopia mineralógica confirmada na base oficial da NASA JPL, e somente **2.94%** possuem diâmetro físico medido por radar/infravermelho!
   - **Implicação Estratégica:** Antes de comprometer sondas pesadas de lavra mineral, o maior retorno sobre informação está na operação/venda de dados de **nanossondas precursoras de reconhecimento espectral** para alvos NHATS de baixo $\Delta v$.

---

## 4. Análise de Corredores Orbitais e Critérios de Seleção Multicritério (TOPSIS)

### 4.1 Diagnóstico Empírico dos Corredores de Tráfego Orbital
- **Bifurcação de Risco em LEO:** O corredor `LEO_CONSTELACAO` (450–650 km) abriga **13,483 objetos em órbita**, sendo **12,599 satélites ativos manobráveis** (5.2% detritos). Em contraste, `LEO_SSO_ALTA` (650–2.000 km) possui volume similar (**13,511 objetos**), porém composto por **74.5% de detritos e estágios inertes (10,072 objetos não manobráveis)** — índice de congestão **89.8/100**.
- **Alerta Crítico no Corredor de Escape `HEO_CISLUNAR`:** Das **1,503 estruturas em órbita** no corredor elíptico/cislunar obrigatório para missões a asteroides, **84.0% (1,262 objetos)** são estágios superiores descartados (`R/B`) e detritos cruzando o perigeu terrestre. As janelas de injeção translunar/interplanetária devem evitar estritamente estacionamento prolongado acima de 650 km.

### 4.2 Funil de Seleção de Dados Aplicado ao Portfólio
| etapa_funil                                            |   candidatos |   retencao_global_pct |
|:-------------------------------------------------------|-------------:|----------------------:|
| 1. Catálogo Global NASA SBDB (NEOs)                    |        43072 |               100     |
| 2. Solução Balística Viável (NASA NHATS)               |         7169 |                16.644 |
| 3. Janela Energética (Δv ≤ 6.5 km/s & Duração ≤ 450 d) |         1042 |                 2.419 |
| 4. Qualidade Informacional (OCC ≤ 3 & Obs ≥ 50)        |          174 |                 0.404 |
| 5. Porte Físico Operacional (Diâmetro Ref ≥ 25 m)      |           92 |                 0.214 |
| 6. Portfólio Qualificado Final (Pós-Filtro Espectral)  |           92 |                 0.214 |

### 4.3 Top 10 Asteroides Recomendados para Prospecção (Ranking MCDA-TOPSIS)
|   ranking_topsis | pdes       | nome_completo            | classe_espectral    |   min_delta_v_kms |   duracao_min_dv_dias |   diametro_fisico_ref_m |   num_observacoes |   codigo_incerteza_occ |   score_topsis |
|-----------------:|:-----------|:-------------------------|:--------------------|------------------:|----------------------:|------------------------:|------------------:|-----------------------:|---------------:|
|                1 | 99942      | 99942 Apophis (2004 MN4) | Sq                  |             6.049 |                   354 |                340      |              7370 |                      0 |          60.62 |
|                2 | 2009 HC    | (2009 HC)                | NAO_ESPECTROGRAFADO |             4.44  |                   362 |                 42.6615 |               209 |                      0 |          57.25 |
|                3 | 2014 MF18  | (2014 MF18)              | NAO_ESPECTROGRAFADO |             6.269 |                   202 |                 69      |                78 |                      0 |          54.1  |
|                4 | 341843     | 341843 (2008 EV5)        | NAO_ESPECTROGRAFADO |             6.291 |                   362 |                400      |              1486 |                      0 |          51.52 |
|                5 | 2006 BZ147 | (2006 BZ147)             | NAO_ESPECTROGRAFADO |             4.091 |                   362 |                 30.1662 |                64 |                      3 |          50.56 |
|                6 | 2024 BF    | (2024 BF)                | NAO_ESPECTROGRAFADO |             4.546 |                   362 |                 46.1952 |                82 |                      2 |          46.42 |
|                7 | 478784     | 478784 (2012 UV136)      | NAO_ESPECTROGRAFADO |             4.67  |                   394 |                 27.6948 |               148 |                      0 |          46.39 |
|                8 | 2020 MU1   | (2020 MU1)               | NAO_ESPECTROGRAFADO |             4.403 |                   370 |                 40.1871 |                54 |                      2 |          45.58 |
|                9 | 2023 KX3   | (2023 KX3)               | NAO_ESPECTROGRAFADO |             4.17  |                   370 |                 25.6905 |                80 |                      3 |          45.25 |
|               10 | 613862     | 613862 (2007 UY1)        | NAO_ESPECTROGRAFADO |             5.579 |                   362 |                 88.6228 |               312 |                      0 |          44.37 |

### 4.4 Veículos Lançadores Qualificados (Critério Atuarial Wilson 95% & Atividade NewSpace)
| modelo_veiculo   | agencia_principal   |   total_lancamentos |   taxa_sucesso_pct |   wilson_lower_95_pct |   ultimo_voo_ano |
|:-----------------|:--------------------|--------------------:|-------------------:|----------------------:|-----------------:|
| Falcon 9         | SPX                 |                 699 |              99.57 |                 98.75 |             2026 |
| Chang Zheng 2D   | SAST                |                 102 |              99.02 |                 94.65 |             2026 |
| Chang Zheng 2C   | CALT                |                  74 |              98.65 |                 92.73 |             2026 |
| Soyuz-FG         | RAKA                |                  70 |              98.57 |                 92.34 |             2019 |
| Chang Zheng 4B   | SAST                |                  59 |              98.31 |                 91    |             2026 |

### 4.5 Cenários Integrados de Subscrição Atuarial (Alvo $\times$ Lançador $\times$ Corredor Cislunar)
| id_cenario   | nome_asteroide           | modelo_veiculo   |   min_delta_v_kms |   indice_maturidade_info |   score_topsis |   score_risco_combinado | rating_atuarial   |
|:-------------|:-------------------------|:-----------------|------------------:|-------------------------:|---------------:|------------------------:|:------------------|
| NS_SCEN_001  | 99942 Apophis (2004 MN4) | Falcon 9         |             6.049 |                    98.89 |          60.62 |                   24.07 | A                 |
| NS_SCEN_002  | 99942 Apophis (2004 MN4) | Chang Zheng 2D   |             6.049 |                    98.89 |          60.62 |                   31.24 | BB                |
| NS_SCEN_003  | 99942 Apophis (2004 MN4) | Chang Zheng 2C   |             6.049 |                    98.89 |          60.62 |                   34.6  | BB                |
| NS_SCEN_004  | (2009 HC)                | Falcon 9         |             4.44  |                    69.67 |          57.25 |                   29.59 | A                 |
| NS_SCEN_005  | (2009 HC)                | Chang Zheng 2D   |             4.44  |                    69.67 |          57.25 |                   36.76 | BB                |
| NS_SCEN_006  | (2009 HC)                | Chang Zheng 2C   |             4.44  |                    69.67 |          57.25 |                   40.12 | B                 |
| NS_SCEN_007  | (2014 MF18)              | Falcon 9         |             6.269 |                    67.59 |          54.1  |                   29.46 | A                 |
| NS_SCEN_008  | (2014 MF18)              | Chang Zheng 2D   |             6.269 |                    67.59 |          54.1  |                   36.63 | BB                |

---

## 5. Plano Diretor de Recomendações para Tomada de Decisão

1. **Adoção de Arquitetura Híbrida de Dados (SQLite WAL + Parquet/DuckDB):** Manter o banco relacional indexado (`newspace_analytics.db`) como repositório transacional auditável (chaves estrangeiras ativas) acoplado a camadas colunares Parquet consultadas via DuckDB/Polars para sustentar o crescimento projetado até 2030 sem degradação de SLA no dashboard.
2. **Estratégia em Duas Fases para Prospecção Espacial:**
   - **Fase Alfa (Monetização de Informação Mineralógica):** Priorizar alvos no topo do ranking TOPSIS com $\Delta v \le 5.5\text{ km/s}$ e `OCC = 0` que ainda figuram como `NAO_ESPECTROGRAFADO`, comercializando dados proprietários de composição espectral e radar para agências e consórcios de recursos espaciais.
   - **Fase Beta (Missão de Extração / Demonstração ISRU):** Direcionar missões dedicadas exclusivamente a alvos já caracterizados nos complexos C (água/propelente orbital) e S/X (metais) com `OCC <= 1` e centenas de observações astrométricas.
3. **Diretriz de Subscrição Securitária Aeroespacial:** Restringir apólices de Rating `AA`/`A` a veículos lançadores ativos com limite inferior de confiabilidade de Wilson (95%) $\ge 95\%$, exigindo órbita de estacionamento inicial em `LEO_VLEO` ($< 450\text{ km}$) antes da queima de injeção interplanetária para minimizar tempo de exposição aos detritos de `LEO_SSO_ALTA` e `HEO_CISLUNAR`.
